# Notebook 2 — Data: inputs, configuration, and outputs

This notebook walks through the data the project consumes and produces, then reads each canonical output from disk. It also tours the online portals and repositories where the external datasets can be downloaded, with screenshots of each. These are the files the web application serves.


## Learning objectives

- Map the external inputs to `src/model/config.yaml`.
- Tour the online data portals and repositories behind each input (bathymetry, land mask, tidal forcing).
- Name the six canonical output files and where they live.
- Read the power raster, the NetCDF time series, and the hotspot GeoJSON.


## 2.1 Inputs

The screening model needs three kinds of external input, all declared in `src/model/config.yaml`:

| Input | Source | Role |
|-------|--------|------|
| Bathymetry | **GEBCO 2026** NetCDF | Seabed depth *h* (m, positive down) |
| Tidal harmonics | **GOT4.10c** / FES2014 / TPXO9 (or synthetic) | M2/S2/K1/O1 amplitudes & phases |
| Land mask | **Philippines landmass GeoJSON** (GeoBoundaries ADM0) | Mark dry cells |

The next section visits the websites that publish these datasets. Every one of them is free to download (some require a one-time registration). `downloader.py` automates the ones with direct URLs; the rest are documented so you can fetch them by hand.


In [1]:
try:
    from model.config import load_config
    cfg = load_config()
    print('domain        :', cfg['domain'])
    print('bathymetry    :', cfg['bathymetry'].get('path'))
    print('land mask     :', cfg['bathymetry'].get('land_shapefile'))
    print('forcing source:', cfg['tidal_forcing'].get('source'))
    print('constituents  :', cfg['tidal_forcing'].get('constituents'))
    print('duration_days :', cfg['simulation'].get('duration_days'))
    print('resolution_km :', cfg['domain'].get('resolution_km'))
    print('hotspot thr.  :', cfg['output'].get('hotspot_threshold'), 'W/m^2')
    print('engine        :', cfg['engine'].get('name'))
except Exception as exc:
    print('config not loaded:', exc)


domain        : {'lon_min': 116.0, 'lon_max': 128.0, 'lat_min': 4.0, 'lat_max': 22.0, 'resolution_km': 2.0}
bathymetry    : data/GEBCO_28_Aug_2026_0bcab7925fa2/gebco_2026_n22.0_s4.0_w112.0_e128.0.nc
land mask     : data/philippines_landmass.geojson
forcing source: got
constituents  : ['M2', 'S2', 'K1', 'O1']
duration_days : 15
resolution_km : 2.0
hotspot thr.  : 200.0 W/m^2
engine        : python


## 2.2 The online data landscape

Tidal-energy resource assessment is data-hungry. You need a terrain model for the seabed, a coastline to separate water from land, and a tidal model to know how high and how fast the water moves. This section surveys the leading free repositories for each, in the same order the model consumes them. Screenshots were captured with Playwright; the capture script lives at `scripts/capture_portal_screenshots.py`.

### 2.2.1 Bathymetry — GEBCO and friends

#### GEBCO — the global grid

GEBCO (General Bathymetric Chart of the Oceans) is an international programme that assembles the global GEBCO Grid — a continuous **15 arc-second (~450 m)** terrain model covering both oceans and land, published annually (GEBCO 2024, 2025, 2026, ...). It is the standard, freely available bathymetry used in marine energy screening studies worldwide. The grid is distributed as a global NetCDF (~7 GB), as eight 90°×90° GeoTIFF tiles, and in Esri ASCII; the site also documents the licence terms and the recommended citation (doi:10.5285/4f68d5c7-45eb-f999-e063-7086abc036fa).

In this project: `data/gebco/gebco_2026_n22.0_s4.0_w112.0_e128.0.nc` is a Philippine-region subset of the GEBCO 2026 grid, read by `bathymetry.load_gebco`.

![GEBCO gridded bathymetry data page](images/portal-gebco.png)


#### GEBCO Download App (subsetting)

Because the full global grid is huge, GEBCO provides an interactive **Download App** where you draw a bounding box on a map and download only that region in NetCDF, GeoTIFF, or Esri ASCII. You can add several regions to a basket and receive a zip of processed subsets. This is the fastest way to grab a modest area like the Philippines by hand.

In this project: the old app's output filename scheme (`gebco_2026_n<latmax>_s<latmin>_w<lonmin>_e<lonmax>.nc`) is kept verbatim by `downloader.py` so files produced here and by the script are interchangeable.

![GEBCO subsetting app](images/portal-gebco-subset.png)


#### GEBCO on CEDA — the official archive

The authoritative direct-download home of the GEBCO grids is the Centre for Environmental Data Analysis (**CEDA**), part of the UK Natural Environment Research Council. It archives the global NetCDF, the GeoTIFF tiles, the Type-Identifier (TID) grid, and offers OPeNDAP access for programmatic subsetting. The global ice-surface NetCDF is ~7 GB — fine for a one-off download, but overkill when you only need one archipelago.

In this project: the COG mirror in §2.2.1 (below) reads the same underlying grid, so you normally do not need to visit CEDA.

![GEBCO 2026 on CEDA](images/portal-gebco-ceda.png)


#### GEBCO as a Cloud-Optimized GeoTIFF (source.coop)

data.source.coop is a community-run data cooperative that re-publishes popular geospatial datasets as **Cloud-Optimized GeoTIFFs (COGs)** for fast, range-based access. This mirror of GEBCO 2026 (`giswqs/gebco-bathymetry`) lets a client fetch only the tiles it needs over plain HTTP — no 7 GB download. The COG carries the original 15 arc-second values plus internal overviews for fast zooming.

In this project: `downloader.py --gebco` opens this COG with `rasterio` and reads just the Philippines window (~17 MB) before writing the GEBCO-format NetCDF. This is the automated path you will actually use.

![GEBCO COG mirror on source.coop](images/portal-gebco-cog.png)


#### EMODnet Bathymetry (regional complement)

EMODnet Bathymetry is a European Commission portal that harmonizes thousands of national surveys into a high-resolution Digital Terrain Model for European waters, filling gaps with GEBCO. It provides a DTM viewer, WMS/WFS services, and free downloads. It is a good example of a survey-graded regional product you might swap in for the global grid in a European study.

In this project: not used (Philippine domain), but a useful reference for how regional surveys upgrade global grids.

![EMODnet Bathymetry portal](images/portal-emodnet.png)


### 2.2.2 Land boundaries — where the coast is

#### GeoBoundaries

geoBoundaries is a free, open-source repository of administrative boundaries (ADM0–ADM5) for every country, maintained by the GeoLab at William & Mary. It exposes a REST API (`api.geoboundaries.org`) that returns download links for GeoJSON, Shapefile, and TopoJSON, and is updated from official sources. It is lightweight, licence-permissive, and perfect for grabbing a country outline as a land mask.

In this project: `data/philippines_landmass.geojson` is the Philippines ADM0 outline from geoBoundaries, downloaded by `downloader.py --landmask` and rasterised by `bathymetry.build_land_mask`.

![GeoBoundaries](images/portal-geoboundaries.png)


#### GADM

GADM is a widely used, high-resolution database of country administrative areas, with polygon and line layers from country level down to level 5. Each country can be downloaded free as a shapefile or GeoPackage, and the dataset is a common source of `ADM0` country polygons in GIS workflows. It is a solid alternative to GeoBoundaries for building the Philippine land mask.

In this project: an alternative land-mask source — the country outline layer (`gadm41_PHL_0`) is equivalent to the GeoBoundaries ADM0 used by default.

![GADM](images/portal-gadm.png)


#### OpenStreetMap via Geofabrik

Geofabrik is a company that publishes free, regularly updated extracts of OpenStreetMap for continents, countries, and regions. Its country pages offer raw OSM data as well as a 'free' shapefile bundle — including water polygons, coastlines, and land polygons — ready for GIS use. The coastline/land-polygon layers are a well-known way to derive a fine-scale land mask.

In this project: an alternative coastline source for the land mask, often at higher spatial fidelity than ADM0 outlines.

![Geofabrik Philippines page](images/portal-geofabrik.png)


### 2.2.3 Tidal forcing — global ocean tide models

#### NASA GSFC — GOT4.10c (Goddard Ocean Tide)

NASA Goddard's Geodesy and Geophysics Laboratory publishes the **GOT** family of global ocean tide models, derived from decades of satellite altimetry (Topex/Poseidon, Jason, ...). GOT4.10c is a freely downloadable, **no-registration** archive of per-constituent grids (M2, S2, K1, O1, ...) in ASCII and NetCDF, on a 0.5° grid. It is the recommended tidal forcing for this workshop because it is both free and simple to automate.

In this project: the default forcing — `data/GOT4.10c/grids_oceantide_netcdf/*.nc` is read by `forcing.read_got_constituents`.

![NASA GSFC ocean tide models](images/portal-nasa-got.png)


#### AVISO — FES2014

FES2014 (Finite Element Solution) is a global tide model developed by NOVELTIS/LEGOS/CLS and distributed by **AVISO**, the altimetry data centre. It provides amplitude and phase for about 34 constituents on a 1/16° grid, the finest of the three models covered here. Access requires a free AVISO registration, so the downloader can only document it, not automate it.

In this project: an alternative forcing — per-constituent files (`M2_ocean.nc`, `M2_load.nc`, ...) would be placed in `data/fes2014/` and selected with `tidal_forcing.source: fes2014`.

![AVISO FES2014](images/portal-aviso-fes.png)


#### Oregon State University — TPXO9-atlas

TPXO is Oregon State University's series of global barotropic ocean tide models; **TPXO9-atlas** is the latest high-resolution release (~1/30°) with around 90 tidal constituents in one NetCDF. The portal hosts the grids and an interactive viewer; downloads require a simple registration. It is the highest-resolution tide model the project's forcing reader supports.

In this project: an alternative forcing — the single grid file (`h_tpxo9.v1.nc`) would be placed in `data/tpxo9/` and selected with `tidal_forcing.source: tpxo9`.

![TPXO9-atlas](images/portal-tpxo.png)


#### pyTMD — programmatic access to all of the above

pyTMD is an open-source Python library (NASA/JPL, Tyler Sutterley) that downloads, reads, and evaluates ocean tide models including GOT, FES, TPXO, and more. Its `fetch_gsfc_got()` function can automate the GOT4.10c archive download used here, and its readers understand the same per-constituent NetCDF format. It is a handy reference implementation if you ever need constituents outside the M2/S2/K1/O1 set.

In this project: not required, but a useful cross-check for the harmonic values the model interpolates.

![pyTMD documentation](images/portal-pytmd.png)


### 2.2.4 The downloader manifest

All of this is wired into `downloader.py`, which mirrors exactly the files `config.yaml` expects. The cell below prints the live URL manifest so you can see the mapping from repository to download command at a glance.


In [2]:
import sys
from pathlib import Path

root = next((c for c in (Path('.'), Path('../..'))
             if (c / 'downloader.py').exists()), Path('.'))
sys.path.insert(0, str(root))

try:
    import downloader as dl
    for key, ds in dl.DATASETS.items():
        urls = list(ds.get('urls') or [])
        if ds.get('fetcher') == 'gebco_subset':
            urls.append(dl.GEBCO_COG_URL)
        print(f'{key:12s}  {ds["name"]}')
        for u in urls:
            print(f'          {u}')
except Exception as exc:
    print('downloader module not importable:', exc)


gebco         GEBCO 2026 bathymetry (Philippines subset)
          https://data.source.coop/giswqs/gebco-bathymetry/gebco_2026/gebco_2026.tif
landmask      Philippines landmass (GeoBoundaries ADM0)
          https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/gbOpen/PHL/ADM0/geoBoundaries-PHL-ADM0_simplified.geojson
got           GOT4.10c tidal constituents (NASA GSFC)
          https://earth.gsfc.nasa.gov/sites/default/files/2023-12/got4.10c.tar.gz
fes2014       FES2014 tidal constituents
tpxo9         TPXO9-atlas tidal constituents


| Input | Repository / portal | Config key | Download |
|-------|---------------------|------------|----------|
| Bathymetry | GEBCO 2026 — gebco.net · CEDA · source.coop COG | `bathymetry.path` | `--gebco` |
| Land mask | GeoBoundaries ADM0 · GADM · OSM/Geofabrik | `bathymetry.land_shapefile` | `--landmask` |
| Tidal forcing | NASA GOT4.10c · AVISO FES2014 · TPXO9 | `tidal_forcing.path` | `--tidal` |

Run `python downloader.py --all` to fetch the three automated datasets; FES2014 and TPXO9 print manual instructions because they require registration.


## 2.3 Path helpers

The notebooks can be run from the repository root **or** from `docs/workshop/`. These helpers find files relative to either location.


In [3]:
from pathlib import Path

ROOTS = (Path('.'), Path('../..'))   # candidate repo roots

def repo_root():
    return next((r for r in ROOTS if (r / 'src').is_dir()), Path('.'))

def first_that_exists(*names):
    for r in ROOTS:
        for n in names:
            c = r / n
            if c.exists():
                return c
    return None


## 2.4 Outputs — the canonical contract

Both hydrodynamic engines write the **same six files**, so the web layer never cares which engine produced them:

| File | Contents | Units |
|------|----------|-------|
| `tidal_power_density.tif` | Time-mean power density (primary product) | W/m² |
| `max_current_speed.tif` | Max depth-averaged speed | m/s |
| `bathymetry.tif` | Bathymetric depth | m |
| `distance_to_coast.tif` | Distance to nearest coast | km |
| `results.nc` | Time series (η, u, v, power) | mixed |
| `hotspots.geojson` | Ranked sites ≥ threshold | W/m², m |

Screening writes them to `output/`; a TELEMAC refinement writes the same names under `output/telemac/<region>/`.


In [4]:
d = repo_root() / 'output'
if d.is_dir():
    print(f'[{d.resolve()}]')
    for f in sorted(d.iterdir()):
        if f.is_file():
            print(f'  {f.name:<28s} {f.stat().st_size / 1e6:8.2f} MB')
    for sub in ('telemac', 'screenshots'):
        s = d / sub
        if s.is_dir():
            print(f'  {sub}/ ({len(list(s.iterdir()))} entries)')
else:
    print('output/ not present — run the screening model first')


[/Users/lkpanganiban/Dev/fun/tidal-oss/output]
  .DS_Store                        0.02 MB
  .gitkeep                         0.00 MB
  bathymetry.tif                   3.16 MB
  distance_to_coast.tif            3.17 MB
  hotspots.geojson                 0.47 MB
  max_current_speed.tif            3.38 MB
  results.nc                    2707.89 MB
  tidal_power_density.tif          3.61 MB
  tidal_power_density.tif.aux.xml     0.00 MB
  telemac/ (4 entries)
  screenshots/ (8 entries)


## 2.5 Read the power raster

The GeoTIFFs are Cloud-Optimised, EPSG:4326, float32, with NaN nodata for land cells.


In [5]:
import numpy as np

try:
    import rasterio
except ImportError:
    print('rasterio not installed — install it to read GeoTIFFs')
else:
    p = first_that_exists('output/tidal_power_density.tif')
    if p is None:
        print('no screening raster found (output/tidal_power_density.tif)')
    else:
        with rasterio.open(p) as src:
            arr = src.read(1, masked=True)
            print('source:', p)
            print('bounds:', src.bounds)
            print('crs   :', src.crs)
            print('shape :', arr.shape)
            v = arr.compressed()
            if v.size:
                print(f'min   : {v.min():.1f} W/m^2')
                print(f'mean  : {v.mean():.1f} W/m^2')
                print(f'max   : {v.max():.1f} W/m^2')
                print(f'P95   : {np.percentile(v, 95):.1f} W/m^2')
                print(f'cells >= 200 W/m^2: {int((arr >= 200).sum())}')


source: ../../output/tidal_power_density.tif
bounds: BoundingBox(left=116.0020833333333, bottom=4.002083333333331, right=127.99791666666664, top=21.99791666666667)
crs   : EPSG:4326
shape : (1002, 668)
min   : nan W/m^2
mean  : nan W/m^2
max   : nan W/m^2
P95   : nan W/m^2
cells >= 200 W/m^2: 1559


## 2.6 Read the NetCDF time series

`results.nc` holds hourly snapshots of η, staggered u/v, and power. We extract the time series at the raster cell nearest a chosen coordinate.


In [6]:
try:
    from netCDF4 import Dataset
except ImportError:
    print('netCDF4 not installed')
else:
    p = first_that_exists('output/results.nc')
    if p is None:
        print('no results.nc found')
    else:
        with Dataset(p) as nc:
            lat = np.asarray(nc['lat'])
            lon = np.asarray(nc['lon'])
            tgt = (12.5, 122.5)
            d2 = (lat - tgt[0]) ** 2 + (lon - tgt[1]) ** 2
            r, c = np.unravel_index(np.argmin(d2), lat.shape)
            times_h = np.asarray(nc['time']) / 3600.0
            eta = np.asarray(nc['eta'][:, r, c])
            pwr = np.asarray(nc['power_density'][:, r, c])
            print('source      :', p)
            print('nearest cell:', round(float(lat[r, c]), 3),
                  round(float(lon[r, c]), 3))
            print('window      : %.1f h, %d samples'
                  % (times_h[-1] - times_h[0], times_h.size))
            print(f'max |eta|    : {np.abs(eta).max():.2f} m')
            print(f'mean power   : {pwr.mean():.1f} W/m^2')


source      : ../../output/results.nc
nearest cell: 12.506 122.495
window      : 360.0 h, 361 samples
max |eta|    : 1.22 m
mean power   : 0.0 W/m^2


## 2.7 Read the hotspot GeoJSON

`hotspots.geojson` is a `FeatureCollection` of points at cell centres with power density and depth properties.


In [7]:
import json

p = first_that_exists('output/hotspots.geojson')
if p is None:
    print('no hotspots.geojson found')
else:
    fc = json.loads(p.read_text())
    print('source  :', p)
    print('features:', len(fc['features']))
    for f in fc['features'][:5]:
        pr = f['properties']
        print(f"  {f['geometry']['coordinates'][0]:9.3f}, "
              f"{f['geometry']['coordinates'][1]:7.3f}  "
              f"{pr['power_density_Wm2']:7.1f} W/m^2  "
              f"depth={pr['depth_m']:.0f} m")


source  : ../../output/hotspots.geojson
features: 1559
    117.855,   4.002    258.0 W/m^2  depth=13 m
    117.872,   4.002    240.1 W/m^2  depth=10 m
    117.926,   4.002    249.5 W/m^2  depth=9 m
    117.944,   4.002    361.5 W/m^2  depth=8 m
    117.962,   4.002    534.7 W/m^2  depth=14 m


## Next

You can read every output and you know where the inputs come from. Move to [Notebook 3 — general workflow](3.general-workflow.ipynb) to see how they are produced end to end.

---

[Index](README.md) · [← 1.concept.ipynb](1.concept.ipynb) · [3.general-workflow.ipynb →](3.general-workflow.ipynb)
